# Figure 5B / 6A / 6B — Cell type enrichment in MHC II high vs low tumors

**Goal:** Identify cell types enriched in MHC II high vs low LUAD tumors
using the Salcher atlas tumor primary samples. Cell type abundance is
computed per sample, then compared between MHC II classifications using
Mann-Whitney U with BH FDR correction.

**Input:**
- `outputs/processed/luad_mhc2_classified.h5ad` — all tumor primary cells
  with MHC II classification derived from cancer cell clustering
- `outputs/processed/luad_epithelial_harmony.h5ad` — epithelial subset
  (source of MHC II classification labels)

**Output:** Figure 5B (lymphocyte enrichment) and Figure 6A/B (APC/all
cell type enrichment) saved to `outputs/figures/`.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
import anndata as ad
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests
from pathlib import Path
import yaml

sns.set(font_scale=1.6)
sns.set_style('ticks')
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

cmap_high_low = ['#FF8811FF', '#462255FF', '#9DD9D2FF', '#046E8FFF', '#D44D5CFF']
colors = {
    'MHC class II High': cmap_high_low[0],
    'MHC class II Low':  cmap_high_low[1],
}
group_order = ['MHC class II High', 'MHC class II Low']

## Data Loading

In [ ]:
with open('/home/gh8sj/projects/mhc2-luad-paper/data/paths_config.yml') as f:
    cfg = yaml.safe_load(f)

repo_root = Path(cfg['repo_root'])
fig_dir   = repo_root / cfg['outputs']['figures']
table_dir = repo_root / cfg['outputs']['tables']

fig5_out   = fig_dir   / 'figure5'
fig6_out   = fig_dir   / 'figure6'
table_out  = table_dir / 'figure5_6'

fig5_out.mkdir(parents=True, exist_ok=True)
fig6_out.mkdir(parents=True, exist_ok=True)
table_out.mkdir(parents=True, exist_ok=True)


In [ ]:
tumordata = sc.read_h5ad(repo_root / 'outputs/processed/luad_mhc2_classified.h5ad')
print(tumordata)
print(tumordata.obs.columns.tolist())
print(tumordata.obs['cell_type_major'].value_counts())

In [ ]:
# confirm MHC2 classification column name and values
print(tumordata.obs['MHC2_clustering'].value_counts())

## Cell type abundance matrix

Per-sample cell type counts are computed from `cell_type_major` annotations.
Only samples with a confirmed MHC II classification (high or low) are included.
Samples labeled 'Excluded from Clustering' are dropped — these are samples
where the cancer cell MHC II signal was insufficient for reliable classification.

In [ ]:
# restrict to classified samples only
classified = tumordata[
    tumordata.obs['MHC2_clustering'].isin(['MHC class II High', 'MHC class II Low'])
].copy()

# per-sample MHC2 classification
mhc2_map = (
    classified.obs.groupby('sample', observed = True)['MHC2_clustering']
    .first()
    .to_dict()
)

# per-sample cell_type_major counts
ct_col = 'cell_type_major'
counts = (
    classified.obs.groupby(['sample', ct_col], observed = True)
    .size()
    .unstack(fill_value=0)
)
counts['MHC2_clustering'] = counts.index.map(mhc2_map)

print(f"Samples: {len(counts)}")
print(counts['MHC2_clustering'].value_counts())

## Statistical testing — Mann-Whitney U with BH FDR correction

Cell type abundance (raw counts per sample) is compared between MHC II high
and MHC II low samples using Mann-Whitney U tests. FDR correction is applied
across all cell types simultaneously using Benjamini-Hochberg. Raw counts are
used rather than proportions to avoid compositional effects from the tumor
cell compartment dominating normalization.

In [ ]:
cell_cols = [c for c in counts.columns if c != 'MHC2_clustering']

rows = []
for col in cell_cols:
    high = counts.loc[counts['MHC2_clustering'] == 'MHC class II High', col]
    low  = counts.loc[counts['MHC2_clustering'] == 'MHC class II Low',  col]
    _, p = mannwhitneyu(high, low, alternative='two-sided')
    rows.append({
        'Cell Type':    col,
        'p_value':      p,
        'mean_high':    high.mean(),
        'mean_low':     low.mean(),
        'Higher Group': 'MHC class II High' if high.mean() > low.mean() else 'MHC class II Low',
    })

stats_df = pd.DataFrame(rows)
_, fdr, _, _ = multipletests(stats_df['p_value'], method='fdr_bh')
stats_df['FDR_p']        = fdr
stats_df['-log10_FDR_p'] = -np.log10(fdr)
stats_df['Significant']  = fdr < 0.05
stats_df = stats_df.sort_values('-log10_FDR_p', ascending=False)

stats_df.to_csv(table_out / 'cell_type_major_enrichment_stats.csv', index=False)
print(stats_df[['Cell Type', 'Higher Group', 'FDR_p', 'Significant']].to_string(index=False))

## Figure 5B / 6A — Cell type enrichment scatter plot

-log10 FDR-adjusted p-value for each cell type, colored by which group shows
higher mean abundance. Filled large points indicate FDR < 0.05. The dashed
vertical line marks the FDR = 0.05 threshold. This combined figure will be
split into lymphocyte (Figure 5B) and APC (Figure 6A) panels for the final
manuscript.

In [ ]:
impofig, axes = plt.subplots(1, 2, figsize=(14, len(supp_df) * 0.3 + 2), sharey=False)

for ax, (metric, col, sig_col) in zip(axes, [
    ('% cells expressing',          '-log10_FDR_pct',  'sig_pct'),
    ('Mean expression (pos cells)', '-log10_FDR_mean', 'sig_mean'),
]):
    plot_df = supp_df.sort_values(col, ascending=True).copy()

    for _, row in plot_df.iterrows():
        color  = palette['MHC class II High'] if row['direction'] == 'MHC II High' else palette['MHC class II Low']
        sig    = row[sig_col] != ''
        marker = 'o' if sig else 'X'
        size   = 80 if sig else 50

        ax.plot([0, row[col]], [row['gene'], row['gene']],
                color=color, lw=1.5, alpha=0.6)
        ax.scatter(row[col], row['gene'],
                   color=color, s=size, marker=marker,
                   zorder=3, linewidths=1.2)

    ax.axvline(-np.log10(0.05), color='gray', linestyle='--', lw=1)
    ax.set_xlabel('-log10(FDR-adjusted p-value)', fontsize=11)
    ax.set_title(metric, fontsize=12)
    ax.spines[['top', 'right']].set_visible(False)

legend_elements = [
    Line2D([0], [0], marker='o', color='w',
           markerfacecolor=palette['MHC class II High'],
           markersize=10, label='Higher in MHC II High'),
    Line2D([0], [0], marker='o', color='w',
           markerfacecolor=palette['MHC class II Low'],
           markersize=10, label='Higher in MHC II Low'),
    Line2D([0], [0], marker='o', color='gray',
           markerfacecolor='gray', markersize=10, label='FDR < 0.05'),
    Line2D([0], [0], marker='X', color='gray',
           markersize=8, label='ns', linewidth=0),
]
fig.legend(handles=legend_elements, loc='upper center', ncol=4,
           frameon=False, bbox_to_anchor=(0.5, 1.02), fontsize=10)

plt.tight_layout()
plt.savefig(fig_out / 'figS_tumor_cytokine_full_lollipop.pdf', bbox_inches='tight')
plt.show()